# 📥 Notebook 00 — Download Dataset dari Kaggle

Notebook ini mendownload dataset **Pins Face Recognition** langsung dari Kaggle  
dan menyimpannya ke folder `data/raw/` sesuai struktur proyek.

**Urutan pengerjaan:**
```
00_download_dataset.ipynb   ← (ini)
01_EDA_dataset.ipynb
02_training_pipeline.ipynb
03_evaluation.ipynb
04_export_inference.ipynb
```

---
### Yang dibutuhkan:
- File `kaggle.json` (dari Kaggle → Account → Settings → API → Create New Token)
- Google Colab dengan Google Drive sudah login


## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ── Sesuaikan ini jika nama folder berbeda di Drive ──────────────────────────
BASE_DIR = '/content/drive/MyDrive/face_recognition_ewallet'

In [ ]:
import os
if not os.path.exists(BASE_DIR):
    raise FileNotFoundError(
        f"Folder proyek tidak ditemukan di: {BASE_DIR}\n"
        "Pastikan folder 'face_recognition_ewallet' sudah diupload ke MyDrive."
    )

print(f"✅ Google Drive terhubung")
print(f"📁 Project dir : {BASE_DIR}")


## Step 2 — Install Kaggle & Setup Credentials

In [ ]:
# Install kaggle library
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
print("✅ Kaggle library installed")


In [ ]:
# ── Upload kaggle.json ────────────────────────────────────────────────────────
# Cara dapat kaggle.json:
#   1. Buka https://www.kaggle.com
#   2. Klik foto profil → Settings
#   3. Scroll ke bagian "API"
#   4. Klik "Create New Token" → file kaggle.json akan terdownload
#   5. Upload file itu di cell ini

In [ ]:
from google.colab import files
import os, shutil, json

In [ ]:
print("📂 Silakan upload file kaggle.json Anda:")
uploaded = files.upload()

if 'kaggle.json' not in uploaded:
    raise ValueError("File yang diupload bukan 'kaggle.json'. Coba lagi.")

# Validasi isi file
try:
    kaggle_data = json.loads(uploaded['kaggle.json'].decode('utf-8'))
    assert 'username' in kaggle_data and 'key' in kaggle_data
    print(f"✅ kaggle.json valid — username: {kaggle_data['username']}")
except Exception as e:
    raise ValueError(f"Format kaggle.json tidak valid: {e}")

In [ ]:
# Simpan ke lokasi standar kaggle
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_path = os.path.join(kaggle_dir, 'kaggle.json')
shutil.copy('kaggle.json', kaggle_path)
os.chmod(kaggle_path, 0o600)

print(f"✅ Credentials tersimpan di: {kaggle_path}")


## Step 3 — Buat Folder Tujuan

In [ ]:
from pathlib import Path

# Folder tujuan sesuai struktur proyek
RAW_DIR       = Path(BASE_DIR) / 'data' / 'raw'
PROCESSED_DIR = Path(BASE_DIR) / 'data' / 'processed'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 data/raw/       → {RAW_DIR}")
print(f"📁 data/processed/ → {PROCESSED_DIR}")
print("✅ Folder struktur siap")


## Step 4 — Download Dataset dari Kaggle

In [ ]:
import subprocess, os

DATASET_SLUG = 'hereisburak/pins-face-recognition'
DOWNLOAD_DIR = str(RAW_DIR)

print(f"⬇️  Mendownload dataset: {DATASET_SLUG}")
print(f"📥 Tujuan            : {DOWNLOAD_DIR}")
print(f"⚠️  Ukuran dataset    : ~1.5 GB — harap tunggu...")
print()

result = subprocess.run(
    ['kaggle', 'datasets', 'download',
     '-d', DATASET_SLUG,
     '-p', DOWNLOAD_DIR,
     '--unzip'],           # langsung ekstrak, tidak perlu unzip manual
    capture_output=True, text=True
)

print(result.stdout)
if result.returncode != 0:
    print("❌ ERROR:")
    print(result.stderr)
    raise RuntimeError("Download gagal. Cek credentials dan koneksi internet.")

print("✅ Download & ekstraksi selesai!")


## Step 5 — Verifikasi Struktur Folder

In [ ]:
from pathlib import Path
from collections import defaultdict

img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

In [ ]:
# Cek apa yang ada di data/raw/
raw_path = Path(BASE_DIR) / 'data' / 'raw'
all_items = list(raw_path.iterdir())

print(f"📁 Isi folder data/raw/ ({len(all_items)} item):")
for item in sorted(all_items)[:10]:
    print(f"   {'📁' if item.is_dir() else '📄'} {item.name}")
if len(all_items) > 10:
    print(f"   ... dan {len(all_items) - 10} item lainnya")


In [ ]:
# ── Deteksi struktur dataset otomatis ────────────────────────────────────────
# Dataset Kaggle kadang punya nested folder atau flat folder
# Script ini otomatis mendeteksi dan menampilkan struktur yang ditemukan

raw_path  = Path(BASE_DIR) / 'data' / 'raw'
class_dirs = []

for item in sorted(raw_path.iterdir()):
    if item.is_dir():
        # Cek apakah langsung ada gambar di sini
        direct_imgs = [f for f in item.iterdir() if f.suffix.lower() in img_exts]
        # Atau ada subfolder lagi
        subdirs = [d for d in item.iterdir() if d.is_dir()]
        class_dirs.append((item, len(direct_imgs), len(subdirs)))

print(f"\n📊 Ditemukan {len(class_dirs)} folder kelas")
print(f"\nContoh 5 kelas pertama:")
for cdir, n_imgs, n_subs in class_dirs[:5]:
    name = cdir.name.replace('pins_', '') if cdir.name.lower().startswith('pins_') else cdir.name
    print(f"   📁 {name:30s} | {n_imgs} gambar langsung | {n_subs} subfolder")


In [ ]:
# ── Hitung total gambar & distribusi per kelas ────────────────────────────────
class_counts = {}

for class_dir in sorted(raw_path.iterdir()):
    if not class_dir.is_dir():
        continue
    name = class_dir.name.replace('pins_', '') if class_dir.name.lower().startswith('pins_') else class_dir.name
    count = sum(1 for f in class_dir.rglob('*') if f.suffix.lower() in img_exts)
    if count > 0:
        class_counts[name] = count

total_imgs   = sum(class_counts.values())
total_classes = len(class_counts)
counts_list  = sorted(class_counts.values())

print("=" * 45)
print("  DATASET SUMMARY")
print("=" * 45)
print(f"  Total kelas  : {total_classes}")
print(f"  Total gambar : {total_imgs:,}")
print(f"  Min / kelas  : {min(counts_list)}")
print(f"  Max / kelas  : {max(counts_list)}")
print(f"  Rata-rata    : {total_imgs/total_classes:.1f} gambar/kelas")
print(f"  Median       : {sorted(counts_list)[len(counts_list)//2]}")
print("=" * 45)
print()

# Kelas dengan gambar terbanyak dan paling sedikit
sorted_by_count = sorted(class_counts.items(), key=lambda x: x[1])
print("5 Kelas dengan gambar PALING SEDIKIT:")
for name, count in sorted_by_count[:5]:
    print(f"   {name:30s}: {count} gambar")
print()
print("5 Kelas dengan gambar TERBANYAK:")
for name, count in sorted_by_count[-5:][::-1]:
    print(f"   {name:30s}: {count} gambar")


## Step 6 — Visualisasi Distribusi Dataset

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

results_dir = Path(BASE_DIR) / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

counts = sorted(class_counts.values())
names  = [k for k, _ in sorted(class_counts.items(), key=lambda x: x[1])]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart per kelas (sorted)
axes[0].bar(range(len(counts)), counts, color='steelblue', width=1.0)
axes[0].axhline(y=np.mean(counts), color='red', linestyle='--', label=f'Mean ({np.mean(counts):.0f})')
axes[0].set_title('Jumlah Gambar per Kelas (diurutkan)', fontsize=12)
axes[0].set_xlabel('Rank kelas')
axes[0].set_ylabel('Jumlah gambar')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histogram distribusi
axes[1].hist(counts, bins=20, color='salmon', edgecolor='white')
axes[1].axvline(x=np.mean(counts), color='red', linestyle='--', label=f'Mean ({np.mean(counts):.0f})')
axes[1].set_title('Distribusi Jumlah Gambar per Kelas', fontsize=12)
axes[1].set_xlabel('Jumlah gambar')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Pie: kelas berdasarkan bracket jumlah gambar
brackets = {'< 100': 0, '100-149': 0, '150-199': 0, '≥ 200': 0}
for c in counts:
    if c < 100:          brackets['< 100'] += 1
    elif c < 150:        brackets['100-149'] += 1
    elif c < 200:        brackets['150-199'] += 1
    else:                brackets['≥ 200'] += 1

labels_pie = [f'{k}\n({v} kelas)' for k, v in brackets.items() if v > 0]
sizes_pie  = [v for v in brackets.values() if v > 0]
colors_pie = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99'][:len(sizes_pie)]

axes[2].pie(sizes_pie, labels=labels_pie, colors=colors_pie,
            autopct='%1.1f%%', startangle=90)
axes[2].set_title('Proporsi Kelas per Bracket', fontsize=12)

plt.suptitle('Pins Face Recognition — Dataset Analysis', fontsize=14, y=1.02)
plt.tight_layout()

save_path = str(results_dir / 'eda_dataset_distribution.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Plot disimpan ke: {save_path}")


## Step 7 — Tampilkan Sample Gambar

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

raw_path = Path(BASE_DIR) / 'data' / 'raw'

# Pilih 8 kelas acak
sample_class_dirs = random.sample(
    [d for d in raw_path.iterdir() if d.is_dir()],
    min(8, total_classes)
)

fig, axes = plt.subplots(2, 8, figsize=(22, 6))

for col, cls_dir in enumerate(sample_class_dirs):
    class_name = cls_dir.name.replace('pins_', '')[:15]
    all_imgs = [f for f in cls_dir.rglob('*') if f.suffix.lower() in img_exts]

    for row in range(2):
        ax = axes[row][col]
        if row < len(all_imgs):
            try:
                img = Image.open(all_imgs[row]).convert('RGB')
                ax.imshow(img)
            except:
                ax.text(0.5, 0.5, 'Error', ha='center')
            if row == 0:
                ax.set_title(class_name, fontsize=7, pad=2)
        ax.axis('off')

plt.suptitle('Sample Gambar dari Dataset (Raw — sebelum MTCNN)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print("✅ Preview gambar ditampilkan")


## Step 8 — Validasi Final & Konfirmasi Siap Lanjut

In [ ]:
from pathlib import Path

raw_path  = Path(BASE_DIR) / 'data' / 'raw'
proc_path = Path(BASE_DIR) / 'data' / 'processed'

img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
total_raw = sum(1 for f in raw_path.rglob('*') if f.suffix.lower() in img_exts)
total_cls = len([d for d in raw_path.iterdir() if d.is_dir()])

print("=" * 50)
print("  VALIDASI DATASET SELESAI")
print("=" * 50)
print(f"  📁 data/raw/       — {total_cls} kelas, {total_raw:,} gambar")
print(f"  📁 data/processed/ — kosong (akan diisi di Notebook 01)")
print()

if total_raw < 1000:
    print("⚠️  Jumlah gambar terdeteksi sedikit — cek apakah ekstraksi berhasil penuh")
else:
    print("✅ Dataset berhasil didownload & terstruktur dengan benar!")
    print()
    print("🎯 Langkah selanjutnya:")
    print("   Buka → notebooks/01_EDA_dataset.ipynb")
    print("   (MTCNN preprocessing + EDA akan dijalankan di sana)")
print("=" * 50)
